<a href="https://colab.research.google.com/github/santidrelew/tp-mpg/blob/main/notebooks/03_OLS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3 - Regresión con OLS

In [1]:
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    !git clone https://github.com/santidrelew/tp-mpg.git
    %cd tp-mpg

In [2]:
# Importamos todas las librerías necesarias
from pathlib import Path

import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.formula.api as smf


In [3]:
# Cargamos los datasets
PROJECT_DIR = Path().resolve() if EN_COLAB else Path().resolve().parent
DATA_BASE_DIR = PROJECT_DIR / "data" / "base"

df = pd.read_csv(DATA_BASE_DIR / "milan_airbnb_completo.csv")
df_muestra = pd.read_csv(DATA_BASE_DIR / "milan_airbnb_muestra.csv")

In [4]:
TARGET = "log_daily_price"
FEATURES = df.columns.drop([TARGET, "daily_price"])

X, y = df[FEATURES], df[TARGET]

X_muestra, y_muestra = df_muestra[FEATURES], df_muestra[TARGET]

In [5]:
# Variables categóricas
categoricas = ["neighbourhood_cleansed", "bed_type"]

# Construir la fórmula automáticamente
terminos = [
    f"C({col})" if col in categoricas else col
    for col in FEATURES
]

formula = f"{TARGET} ~ " + " + ".join(terminos)

print(formula)

log_daily_price ~ host_is_superhost + host_total_listings_count + C(neighbourhood_cleansed) + latitude + longitude + accommodates + bathrooms + bedrooms + beds + C(bed_type) + guests_included + minimum_nights + availability_365 + number_of_reviews + review_scores_rating + TV + WiFi + Air_Condition + Wheelchair_accessible + Kitchen + Breakfast + Elevator + Heating + Washer + Iron + Host_greets_you + Paid_parking_on_premises + Luggage_dropoff_allowed + Long_term_stays_allowed + Doorman + Pets_allowed + Smoking_allowed + Suitable_for_events + check_in_24_hour


In [6]:
print(f"Dataset completo: {len(df)} observaciones disponibles")

# Ajustar modelo OLS
modelo_ols = smf.ols(
    formula=formula,
    data=df
).fit()

# Resumen estadístico
print(modelo_ols.summary())

# Predicciones]
y_pred = modelo_ols.predict(df)

# Métricas
print(f"R²:   {r2_score(y, y_pred):.4f}")
print(f"MAE:  {mean_absolute_error(y, y_pred):.4f}")
print(f"RMSE: {mean_squared_error(y, y_pred)**0.5:.4f}")

Dataset completo: 9322 observaciones disponibles
                            OLS Regression Results                            
Dep. Variable:        log_daily_price   R-squared:                       0.400
Model:                            OLS   Adj. R-squared:                  0.397
Method:                 Least Squares   F-statistic:                     140.7
Date:                Sat, 07 Nov 2026   Prob (F-statistic):               0.00
Time:                        11:36:43   Log-Likelihood:                -5298.1
No. Observations:                9322   AIC:                         1.069e+04
Df Residuals:                    9277   BIC:                         1.101e+04
Df Model:                          44                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------

In [7]:
# OLS aplicando filtros y la selección de variables del EDA
df_filtrado = pd.read_csv(DATA_BASE_DIR / "milan_airbnb_completo_filtrado.csv")
df_muestra_filtrada = pd.read_csv(
    DATA_BASE_DIR / "milan_airbnb_muestra_filtrada.csv"
)

terminos_eda = [
    f"C({col})" if col in categoricas else col
    for col in df_filtrado.columns.drop([TARGET, "daily_price"])
]
formula_eda = f"{TARGET} ~ " + " + ".join(terminos_eda)
ajustes_filtrados = {}

for nombre, datos in [
    ("Base completa filtrada", df_filtrado),
    ("Muestra estratificada", df_muestra_filtrada),
]:
    print(f"\n{nombre}: {len(datos)} observaciones")

    modelo = smf.ols(formula=formula_eda, data=datos).fit()
    ajustes_filtrados[nombre] = modelo

    print(modelo.summary())

    y_pred = modelo.predict(datos)
    y_real = datos[TARGET]

    print(f"R²:   {r2_score(y_real, y_pred):.4f}")
    print(f"MAE:  {mean_absolute_error(y_real, y_pred):.4f}")
    print(f"RMSE: {mean_squared_error(y_real, y_pred)**0.5:.4f}")


Base completa filtrada: 9297 observaciones
                            OLS Regression Results                            
Dep. Variable:        log_daily_price   R-squared:                       0.398
Model:                            OLS   Adj. R-squared:                  0.396
Method:                 Least Squares   F-statistic:                     175.0
Date:                Sat, 07 Nov 2026   Prob (F-statistic):               0.00
Time:                        11:36:43   Log-Likelihood:                -5298.8
No. Observations:                9297   AIC:                         1.067e+04
Df Residuals:                    9261   BIC:                         1.093e+04
Df Model:                          35                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------

Comparación de la base filtrada y la muestra:
El R² ajustado es 0.3958 en la base filtrada y 0.4607 en la muestra. Son medidas de ajuste dentro de cada conjunto.

Sobre las mismas filas, comparamos los predictores anteriores, el ajuste sin camas y la selección completa del EDA.

In [8]:
# Recuperamos las columnas previas del CSV completo para comparar las mismas filas.
claves = df_filtrado.columns.tolist()
columnas_previas = df.columns.difference(claves).tolist()
referencia = df[claves + columnas_previas].drop_duplicates(subset=claves, keep=False)
formula_sin_beds = f"{TARGET} ~ " + " + ".join(
    termino for termino in terminos if termino != "beds"
)

filas = []
for nombre, datos in [
    ("Base completa filtrada", df_filtrado),
    ("Muestra estratificada", df_muestra_filtrada),
]:
    datos_previos = datos.merge(
        referencia, on=claves, how="left", validate="one_to_one", indicator=True
    )
    if not datos_previos["_merge"].eq("both").all():
        raise ValueError("No se pudo identificar cada observación en la base completa.")
    modelos = {
        "Predictores anteriores": smf.ols(formula, data=datos_previos).fit(),
        "Sin beds": smf.ols(formula_sin_beds, data=datos_previos).fit(),
        "Selección EDA": ajustes_filtrados[nombre],
    }
    for tratamiento, modelo in modelos.items():
        matriz = pd.DataFrame(modelo.model.exog, columns=modelo.model.exog_names)
        predictores = matriz.loc[:, matriz.std().gt(0)]
        escalados = (predictores - predictores.mean()) / predictores.std(ddof=0)
        diseno = np.column_stack([np.ones(len(escalados)), escalados])
        vif_capacidad = variance_inflation_factor(
            diseno, 1 + escalados.columns.get_loc("accommodates")
        )
        filas.append({
            "Datos": nombre,
            "Predictores": tratamiento,
            "R² ajustado": modelo.rsquared_adj,
            "Coef. accommodates": modelo.params["accommodates"],
            "EE accommodates": modelo.bse["accommodates"],
            "VIF accommodates": vif_capacidad,
            "Condición original": modelo.condition_number,
            "Condición escalada": np.linalg.cond(diseno),
        })

comparacion_eda = pd.DataFrame(filas)
display(comparacion_eda.round(4))

,Datos,Predictores,R² ajustado,Coef. accommodates,EE accommodates,VIF accommodates,Condición original,Condición escalada
0,Base completa filtrada,Predictores anteriores,0.3982,0.0581,0.0051,3.2493,1.361996e+06,6.2055
1,Base completa filtrada,Sin beds,0.3968,0.0454,0.0043,2.3214,1.361303e+06,5.6760
2,Base completa filtrada,Selección EDA,0.3958,0.0459,0.0043,2.3101,1.360682e+06,5.6108
3,Muestra estratificada,Predictores anteriores,0.4609,0.0324,0.0174,3.7653,1.345806e+06,6.7918
4,Muestra estratificada,Sin beds,0.4616,0.0353,0.0152,2.8535,1.345757e+06,6.3075
5,Muestra estratificada,Selección EDA,0.4607,0.0311,0.0150,2.7838,1.339554e+06,6.1909


En la base filtrada, al quitar camas el VIF de capacidad baja de 3.25 a 2.32. El R² ajustado pasa de 0.3982 a 0.3968 y queda en 0.3958 al aplicar todos los descartes del EDA. La caída del número de condición después de escalar muestra la influencia de la escala de los predictores.